In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
 
print("=" * 60)
print("FEATURE ENGINEERING — ICD-9 + Medication + Clinical")
print("=" * 60)

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 3, Finished, Available, Finished, False)

FEATURE ENGINEERING — ICD-9 + Medication + Clinical


## Load Silver

In [2]:
df = spark.read.format("delta").table("silver_encounters_clean")
print(f"Input rows: {df.count():,}")

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 4, Finished, Available, Finished, False)

Input rows: 71,518


## ICD-9 Diagnosis Code Categorisation

In [3]:
# The dataset has 15,000+ unique ICD-9 codes in diag_1/2/3.
# Raw codes are useless to ML — the model overfits on codes
# seen rarely in training. Solution: group into 9 clinical
# categories using HCUP Clinical Classifications Software.

def icd9_category(col_name):
    """Map ICD-9 code column → 9 clinical category string."""
    c = F.col(col_name)
    return (
        F.when(c.rlike(r"^(39[0-9]|4[0-5][0-9])"),            "circulatory")
        .when(c.rlike(r"^(25[0-9])"),                          "diabetes")
        .when(c.rlike(r"^(46[0-9]|47[0-9]|49[0-9]|5[0-1][0-9])"), "respiratory")
        .when(c.rlike(r"^(5[2-9][0-9]|6[0-2][0-9])"),         "digestive")
        .when(c.rlike(r"^(14[0-9]|1[5-9][0-9]|2[0-3][0-9])"), "neoplasm")
        .when(c.rlike(r"^(71[0-9]|72[0-9]|73[0-9])"),         "musculoskeletal")
        .when(c.rlike(r"^(58[0-9]|59[0-9]|60[0-4])"),         "genitourinary")
        .when(c.rlike(r"^(80[0-9]|8[1-9][0-9]|9[0-5][0-9])"), "injury")
        .when(c.isNull(),                                       "unknown")
        .otherwise("other")
    )
 
df = (df
    .withColumn("diag1_cat", icd9_category("diag_1"))
    .withColumn("diag2_cat", icd9_category("diag_2"))
    .withColumn("diag3_cat", icd9_category("diag_3"))
    # Is diabetes the PRIMARY admission reason?
    .withColumn("is_diabetes_primary",
        F.when(F.col("diag_1").rlike(r"^25"), 1).otherwise(0))
    # Does any diagnosis involve circulatory disease?
    .withColumn("has_circulatory_dx",
        F.when(
            (F.col("diag1_cat") == "circulatory") |
            (F.col("diag2_cat") == "circulatory") |
            (F.col("diag3_cat") == "circulatory"), 1
        ).otherwise(0))
)
 
print("\nPrimary diagnosis category distribution:")
df.groupBy("diag1_cat").count().orderBy("count", ascending=False).show(10)

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 5, Finished, Available, Finished, False)


Primary diagnosis category distribution:
+---------------+-----+
|      diag1_cat|count|
+---------------+-----+
|    circulatory|21266|
|          other|21137|
|      digestive|10039|
|       diabetes| 5763|
|    respiratory| 4132|
|musculoskeletal| 3894|
|       neoplasm| 2728|
|         injury| 2542|
|        unknown|   17|
+---------------+-----+



In [4]:
display(df)

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ba2a0d7b-ac68-496d-8f26-9e6c02c79353)

## Medication Change Features

In [5]:
# Each medication column indicates dose change during encounter:
#   'Up'     = dose increased
#   'Down'   = dose decreased
#   'Steady' = prescribed, no change
#   'No'     = not prescribed
#
# Clinical insight: medication changes during admission signal
# the patient's condition was actively unstable — a strong
# readmission predictor.
 
DIABETES_MEDS = [
    "metformin","repaglinide","nateglinide","chlorpropamide",
    "glimepiride","glipizide","glyburide","tolbutamide",
    "pioglitazone","rosiglitazone","acarbose","miglitol",
    "troglitazone","tolazamide","insulin",
    "glyburide-metformin","glipizide-metformin"
]
available_meds = [m for m in DIABETES_MEDS if m in df.columns]
print(f"\nMedication columns available: {len(available_meds)}")
 
change_expr = sum(
    F.when(F.col(m).isin(["Up","Down"]), 1).otherwise(0)
    for m in available_meds
)
taken_expr = sum(
    F.when(~F.col(m).isin(["No"]), 1).otherwise(0)
    for m in available_meds
)
 
df = (df
    .withColumn("total_med_changes", change_expr)
    .withColumn("total_meds_taken",  taken_expr)
    # Insulin is the single strongest individual medication predictor
    .withColumn("insulin_changed",
        F.when(F.col("insulin").isin(["Up","Down"]), 1).otherwise(0))
    .withColumn("insulin_increased",
        F.when(F.col("insulin") == "Up", 1).otherwise(0))
)

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 7, Finished, Available, Finished, False)


Medication columns available: 17


In [6]:
display(df)

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2079b68-0821-41a0-b9a7-d55338a72ac6)

## Lab Result Features

In [7]:
# HbA1c (A1C) is the key diabetes management marker.
# If it was NOT tested during admission → clinically suboptimal.
# If it IS high → patient poorly controlled → higher readmit risk.
 
df = (df
    .withColumn("A1C_tested",
        F.when(F.col("A1Cresult") != "None", 1).otherwise(0))
    .withColumn("A1C_high",
        F.when(F.col("A1Cresult").isin([">7",">8"]), 1).otherwise(0))
    .withColumn("A1C_normal",
        F.when(F.col("A1Cresult") == "Normal", 1).otherwise(0))
)
 
if "max_glu_serum" in df.columns:
    df = df.withColumn("glucose_tested",
        F.when(F.col("max_glu_serum").isin(["Norm",">200",">300"]), 1).otherwise(0))
else:
    df = df.withColumn("glucose_tested", F.lit(0))

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 9, Finished, Available, Finished, False)

## Prior Healthcare Utilisation Features

In [8]:
# Prior hospital/ED use is the strongest single feature group
# in published readmission prediction literature. Patients who
# come in frequently are far more likely to return.
 
df = (df
    .withColumn("prior_visits_total",
        F.col("number_inpatient") +
        F.col("number_emergency") +
        F.col("number_outpatient"))
    # Binary: heavy prior user (≥3 visits in past year)
    .withColumn("is_high_prior_use",
        F.when(F.col("prior_visits_total") >= 3, 1).otherwise(0))
    # Any prior inpatient admission?
    .withColumn("has_prior_inpatient",
        F.when(F.col("number_inpatient") > 0, 1).otherwise(0))
    # Any emergency visit?
    .withColumn("has_prior_emergency",
        F.when(F.col("number_emergency") > 0, 1).otherwise(0))
    # Long length of stay (>7 days signals severity)
    .withColumn("is_long_stay",
        F.when(F.col("time_in_hospital") > 7, 1).otherwise(0))
    # Polypharmacy: ≥15 medications = complex patient
    .withColumn("is_polypharmacy",
        F.when(F.col("num_medications") >= 15, 1).otherwise(0))
    # High number of diagnoses
    .withColumn("is_complex_patient",
        F.when(F.col("number_diagnoses") >= 7, 1).otherwise(0))
)

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 10, Finished, Available, Finished, False)

## Encode Categorical Columns for XGBoost

In [9]:
CAT_COLS = ["diag1_cat","diag2_cat","diag3_cat",
            "gender","race","change","diabetesMed"]
available_cats = [c for c in CAT_COLS if c in df.columns]
 
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in available_cats
]
df = Pipeline(stages=indexers).fit(df).transform(df)
print(f"\nEncoded {len(available_cats)} categorical columns")

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 11, Finished, Available, Finished, False)


Encoded 7 categorical columns


## Final feature inventory

In [10]:
FEATURE_COLS = [
    # Clinical utilisation (strongest feature group)
    "time_in_hospital","num_lab_procedures","num_procedures",
    "num_medications","number_diagnoses",
    "number_inpatient","number_emergency","number_outpatient",
    "prior_visits_total","is_high_prior_use",
    "has_prior_inpatient","has_prior_emergency",
    "is_long_stay","is_polypharmacy","is_complex_patient",
    # Patient demographics
    "age_ord",
    # Medication features
    "total_med_changes","total_meds_taken",
    "insulin_changed","insulin_increased",
    # Lab features
    "A1C_tested","A1C_high","A1C_normal","glucose_tested",
    # Diagnosis features
    "is_diabetes_primary","has_circulatory_dx",
    # Encoded categoricals
    "diag1_cat_idx","diag2_cat_idx","diag3_cat_idx",
    "gender_idx","race_idx",
]
available_feats = [f for f in FEATURE_COLS if f in df.columns]
print(f"\nTotal features: {len(available_feats)}")
for f in available_feats:
    print(f"  + {f}")
 
# ── CELL 8: Write Silver Features ─────────────────────────
df.write.format("delta").mode("overwrite") \
   .option("overwriteSchema", "true") \
   .saveAsTable("silver_features")
 
print(f"\n silver_features: {df.count():,} rows, {len(df.columns)} total columns")
print("Proceed to 04_ml_experiment.py")

StatementMeta(, 8fd8084b-ad70-4cf2-8ee2-fe3fe5af3f2c, 12, Finished, Available, Finished, False)


Total features: 31
  + time_in_hospital
  + num_lab_procedures
  + num_procedures
  + num_medications
  + number_diagnoses
  + number_inpatient
  + number_emergency
  + number_outpatient
  + prior_visits_total
  + is_high_prior_use
  + has_prior_inpatient
  + has_prior_emergency
  + is_long_stay
  + is_polypharmacy
  + is_complex_patient
  + age_ord
  + total_med_changes
  + total_meds_taken
  + insulin_changed
  + insulin_increased
  + A1C_tested
  + A1C_high
  + A1C_normal
  + glucose_tested
  + is_diabetes_primary
  + has_circulatory_dx
  + diag1_cat_idx
  + diag2_cat_idx
  + diag3_cat_idx
  + gender_idx
  + race_idx

 silver_features: 71,518 rows, 78 total columns
Proceed to 04_ml_experiment.py


In [ ]:
# Notebook: 03_feature_engineering.py
# Scenario 02: Healthcare Patient Readmission Prediction
# Input:  silver_encounters_clean
# Output: silver_features (35+ engineered features)